# Session 3: Hohmann transfers, and checking them numerically
**A. Real-world problem.** A satellite is released in a low circular orbit and must reach a
higher one. Fuel is the scarcest thing on board, so every m/s of speed change (delta-v) counts.
Today you plan the cheapest simple transfer (the Hohmann transfer), compare it with a
three-burn alternative (bi-elliptic), price a plane change, catch a target by phasing, and
then check every claim by integrating the real motion step by step.
The model: instant burns ("impulses"), two bodies only (point-mass Earth and satellite),
circular coplanar orbits. This is a teaching model, not a mission design tool.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
MU = 3.986004418e14      # m^3/s^2
R_EARTH = 6.371e6        # m

## Step 1: the Hohmann transfer
The transfer ellipse touches the inner circle (radius r1) at perigee and the outer circle
(radius r2) at apogee, so its semi-major axis is a_t = (r1 + r2) / 2. Use vis-viva:
burn 1 raises speed at r1 from the circular speed to the perigee speed of the ellipse;
burn 2 raises it at r2 from the ellipse's apogee speed to the circular speed. The flight takes
half of the ellipse's period.

In [ ]:
def circular_speed(r):
    return np.sqrt(MU / r)


def vis_viva(r, a):
    return np.sqrt(MU * (2 / r - 1 / a))


def period(a):
    return 2 * np.pi * np.sqrt(a**3 / MU)


def hohmann(r1, r2):
    """Hohmann transfer between circular orbits of radii r1 and r2 (m). Speeds in m/s, time in s."""
    a_t = 0.5 * (r1 + r2)
    dv1 = vis_viva(r1, a_t) - circular_speed(r1)
    dv2 = circular_speed(r2) - vis_viva(r2, a_t)
    return {"dv1": dv1, "dv2": dv2, "total": abs(dv1) + abs(dv2), "time": 0.5 * period(a_t), "a_t": a_t}

## Step 2: check it by integrating the motion
We do not just trust the formula. Start on the inner circle, add dv1 along the velocity, and
integrate the two-body equations (RK4, small steps) for the predicted time. If the plan is right
we arrive at radius r2, on the opposite side of the Earth, moving at the apogee speed.

In [ ]:
def accel(r):
    return -MU * r / np.linalg.norm(r) ** 3


def rk4_step(r, v, dt, thrust=None):
    """One RK4 step. thrust: optional function (r, v) -> extra acceleration (m/s^2)."""
    def f(rr, vv):
        a = accel(rr)
        if thrust is not None:
            a = a + thrust(rr, vv)
        return a
    k1v = f(r, v); k1r = v
    k2v = f(r + 0.5 * dt * k1r, v + 0.5 * dt * k1v); k2r = v + 0.5 * dt * k1v
    k3v = f(r + 0.5 * dt * k2r, v + 0.5 * dt * k2v); k3r = v + 0.5 * dt * k2v
    k4v = f(r + dt * k3r, v + dt * k3v); k4r = v + dt * k3v
    return r + dt / 6 * (k1r + 2 * k2r + 2 * k3r + k4r), v + dt / 6 * (k1v + 2 * k2v + 2 * k3v + k4v)


def integrate(r0, v0, t_end, dt=5.0, thrust=None):
    """Integrate from t = 0 to t_end. Returns (times, positions, velocities)."""
    n = int(np.ceil(t_end / dt))
    h = t_end / n
    r, v = np.array(r0, float), np.array(v0, float)
    ts, rs, vs = [0.0], [r.copy()], [v.copy()]
    for i in range(n):
        r, v = rk4_step(r, v, h, thrust)
        ts.append((i + 1) * h); rs.append(r.copy()); vs.append(v.copy())
    return np.array(ts), np.array(rs), np.array(vs)


def check_hohmann(r1, r2, dt=5.0):
    """Fly the Hohmann transfer numerically. Returns arrival radius, arrival speed, arrival angle and the later circle error."""
    h = hohmann(r1, r2)
    r0 = np.array([r1, 0.0])
    v0 = np.array([0.0, circular_speed(r1) + h["dv1"]])
    ts, rs, vs = integrate(r0, v0, h["time"], dt)
    r_end, v_end = rs[-1], vs[-1]
    angle = np.degrees(np.arctan2(r_end[1], r_end[0])) % 360
    # second burn: add dv2 along the velocity, then fly one full lap of the outer circle
    v_new = v_end + h["dv2"] * v_end / np.linalg.norm(v_end)
    lap = 2 * np.pi * r2 / circular_speed(r2)
    _, rs2, _ = integrate(r_end, v_new, lap, dt)
    err = np.max(np.abs(np.linalg.norm(rs2, axis=1) - r2))
    return {"r_end": np.linalg.norm(r_end), "v_end": np.linalg.norm(v_end), "angle": angle, "circle_err": err}

## Step 3: bi-elliptic transfer, a plane change, Oberth, phasing
*Bi-elliptic*: go out to a far radius rb, then to r2, then back to circular: three burns. For
a big enough ratio r2 / r1 it costs less delta-v than Hohmann, but takes much longer.
*Plane change*: turning the velocity direction by an angle di costs 2 v sin(di / 2).
*Oberth*: the energy a burn adds is v dv + dv^2 / 2, so the same dv gives more energy when v is high.

In [ ]:
def bielliptic(r1, r2, rb):
    a1, a2 = 0.5 * (r1 + rb), 0.5 * (rb + r2)
    dv1 = vis_viva(r1, a1) - circular_speed(r1)
    dv2 = vis_viva(rb, a2) - vis_viva(rb, a1)
    dv3 = abs(circular_speed(r2) - vis_viva(r2, a2))
    return {"dv1": dv1, "dv2": dv2, "dv3": dv3, "total": dv1 + dv2 + dv3, "time": 0.5 * period(a1) + 0.5 * period(a2)}


def cost_ratio(ratio, rb_over_r1=1e6):
    """Bi-elliptic total delta-v divided by Hohmann total delta-v, for r2 / r1 = ratio (very far rb)."""
    r1 = 1.0e7
    return bielliptic(r1, ratio * r1, rb_over_r1 * r1)["total"] / hohmann(r1, ratio * r1)["total"]


def crossover_ratio(lo=8.0, hi=16.0):
    """Radius ratio where bi-elliptic (rb very far) starts to beat Hohmann, by bisection."""
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if cost_ratio(mid) > 1:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def plane_change_dv(v, di_deg):
    """Delta-v (m/s) to rotate a velocity of size v (m/s) by di_deg degrees, speed unchanged."""
    return 2 * v * np.sin(np.radians(di_deg) / 2)


def combined_burn_dv(v1, v2, di_deg):
    """Delta-v (m/s) to change speed from v1 to v2 and turn the direction by di_deg in one burn."""
    return np.sqrt(v1**2 + v2**2 - 2 * v1 * v2 * np.cos(np.radians(di_deg)))


def energy_gain(v, dv):
    """Energy per kg (J/kg) gained by adding dv to speed v: change in 0.5 v^2."""
    return 0.5 * (v + dv) ** 2 - 0.5 * v**2

## Step 4: phasing, and why impulses are only a model
*Phasing*: a chaser in a lower orbit has a higher angular rate and gains on a target that is
ahead. We find the catch-up time numerically from simulated angles. And a real engine does
not push for an instant: a long, weak burn wastes some of its push fighting gravity.

In [ ]:
def phasing_time(r_chaser, r_target, lead_deg, dt=10.0):
    """Time (s) until a chaser in the lower orbit reaches the target, which starts lead_deg ahead.

    Both circular orbits are flown with RK4; the time is found where the angle gap first changes sign.
    """
    t_max = 2 * np.pi / (np.sqrt(MU / r_chaser**3) - np.sqrt(MU / r_target**3)) * 1.1
    out = []
    for r, ang in ((r_chaser, 0.0), (r_target, np.radians(lead_deg))):
        pos = np.array([r * np.cos(ang), r * np.sin(ang)])
        vel = circular_speed(r) * np.array([-np.sin(ang), np.cos(ang)])
        ts, rs, _ = integrate(pos, vel, t_max, dt)
        out.append(np.unwrap(np.arctan2(rs[:, 1], rs[:, 0])))
    gap = out[1] - out[0]
    k = int(np.argmax(gap <= 0))
    return float(ts[k - 1] + (ts[k] - ts[k - 1]) * gap[k - 1] / (gap[k - 1] - gap[k]))


def phasing_formula(r_chaser, r_target, lead_deg):
    n1, n2 = np.sqrt(MU / r_chaser**3), np.sqrt(MU / r_target**3)
    return np.radians(lead_deg) / (n1 - n2)


def orbit_apogee(r, v):
    """Apogee radius (m) of the Kepler orbit through position r and velocity v (2D arrays)."""
    rn = np.linalg.norm(r)
    energy = 0.5 * np.dot(v, v) - MU / rn
    a = -MU / (2 * energy)
    evec = ((np.dot(v, v) - MU / rn) * r - np.dot(r, v) * v) / MU
    if energy >= 0:
        return np.inf
    return a * (1 + np.linalg.norm(evec))


def finite_burn(r1, burn_time, accel_ms2, dt=2.0):
    """Burn along the velocity with constant acceleration, starting from the circular orbit of radius r1.
    Returns the apogee radius (m) of the orbit the satellite is left in (inf if it escapes)."""
    r0, v0 = np.array([r1, 0.0]), np.array([0.0, circular_speed(r1)])
    thrust = lambda rr, vv: accel_ms2 * vv / np.linalg.norm(vv)
    _, rs, vs = integrate(r0, v0, burn_time, dt, thrust)
    return orbit_apogee(rs[-1], vs[-1])


def finite_burn_cost(r1, r2, accel_ms2, dt=2.0):
    """Delta-v (m/s) a constant-acceleration burn really needs to reach apogee r2, found by bisection on burn time.
    Compare with the impulsive hohmann(r1, r2)['dv1']; the difference is the gravity loss."""
    lo, hi = 0.0, 3.0 * hohmann(r1, r2)["dv1"] / accel_ms2
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if finite_burn(r1, mid, accel_ms2, dt) < r2:
            lo = mid
        else:
            hi = mid
    return accel_ms2 * 0.5 * (lo + hi)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    r1, r2 = 6778e3, 42164e3
    h = hohmann(r1, r2)
    print(f"LEO 6778 km -> GEO 42164 km: dv1 {h['dv1'] / 1e3:.3f}, dv2 {h['dv2'] / 1e3:.3f}, total {h['total'] / 1e3:.3f} km/s, time {h['time'] / 3600:.3f} h, a_t {h['a_t'] / 1e3:.0f} km")
    c = check_hohmann(r1, r2)
    print(f"integrated: arrival radius {c['r_end'] / 1e3:.1f} km, speed {c['v_end']:.1f} m/s (apogee speed {vis_viva(r2, h['a_t']):.1f}), angle {c['angle']:.2f} deg, later circle error {c['circle_err']:.1f} m")

    for ratio in (5, 10, 11.94, 15, 20, 50):
        print(f"ratio {ratio:5}: bi-elliptic / Hohmann = {cost_ratio(ratio):.4f}")
    print(f"crossover ratio {crossover_ratio():.3f}")
    b = bielliptic(r1, r2, 20 * r2)
    print(f"LEO->GEO bi-elliptic rb = 20 r2: total {b['total'] / 1e3:.3f} km/s, time {b['time'] / 86400:.1f} days")
    r_a, r_b = 6778e3, 6778e3 * 20
    print("ratio 20, rb=100 r1: hohmann", hohmann(r_a, 20 * r_a)["total"], "bielliptic", bielliptic(r_a, 20 * r_a, 100 * r_a)["total"])

    di = 28.5
    vleo, vgeo = circular_speed(r1), circular_speed(r2)
    vapo = vis_viva(r2, h["a_t"])
    print(f"plane change {di} deg: at LEO {plane_change_dv(vleo, di) / 1e3:.3f}, at GEO {plane_change_dv(vgeo, di) / 1e3:.3f}, at transfer apogee {plane_change_dv(vapo, di) / 1e3:.3f} km/s")
    print(f"apogee speed {vapo / 1e3:.3f}, combined apogee burn {combined_burn_dv(vapo, vgeo, di) / 1e3:.3f} vs separate {(abs(vgeo - vapo) + plane_change_dv(vgeo, di)) / 1e3:.3f} km/s")
    print(f"Oberth: 1000 m/s at {vleo:.0f} m/s adds {energy_gain(vleo, 1000) / 1e6:.3f} MJ/kg, at 1000 m/s adds {energy_gain(1000, 1000) / 1e6:.3f} MJ/kg")

    rl, rt = 6778e3, 7178e3
    print(f"phasing: numeric {phasing_time(rl, rt, 30):.1f} s, formula {phasing_formula(rl, rt, 30):.1f} s")

    for acc in (20.0, 2.0, 0.5):
        tb = h["dv1"] / acc
        ra = finite_burn(r1, tb, acc)
        need = finite_burn_cost(r1, r2, acc)
        print(f"finite burn {acc} m/s^2: burn {tb:.0f} s ({tb / period(r1):.2f} LEO laps), apogee {ra / 1e3:.0f} km (impulsive 42164), real dv needed {need:.0f} m/s, extra {need - h['dv1']:.0f} m/s ({(need - h['dv1']) / h['dv1']:.1%})")

    # Figure 1: delta-v ratio vs radius ratio
    rr = np.linspace(2, 40, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(rr, [cost_ratio(x) for x in rr], label="bi-elliptic (far apogee) / Hohmann")
    ax.axhline(1, color="gray", ls=":")
    ax.axvline(crossover_ratio(), color="red", ls="--", label=f"crossover {crossover_ratio():.2f}")
    ax.set_xlabel("radius ratio r2 / r1 (no unit)"); ax.set_ylabel("delta-v ratio (no unit)")
    ax.set_title("When is three burns cheaper than two?"); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT / "bielliptic_vs_hohmann.png", dpi=120)

    # Figure 2: the integrated transfer
    h2 = hohmann(r1, r2)
    ts, rs, vs = integrate([r1, 0], [0, circular_speed(r1) + h2["dv1"]], h2["time"], 10.0)
    th = np.linspace(0, 2 * np.pi, 300)
    fig2, ax2 = plt.subplots(figsize=(5.5, 5.5))
    ax2.fill(R_EARTH / 1e3 * np.cos(th), R_EARTH / 1e3 * np.sin(th), color="#9ecae1")
    ax2.plot(r1 / 1e3 * np.cos(th), r1 / 1e3 * np.sin(th), "k--", lw=0.8)
    ax2.plot(r2 / 1e3 * np.cos(th), r2 / 1e3 * np.sin(th), "k--", lw=0.8)
    ax2.plot(rs[:, 0] / 1e3, rs[:, 1] / 1e3, "r-", label="integrated transfer")
    ax2.plot([r1 / 1e3], [0], "go", label="burn 1"); ax2.plot([-r2 / 1e3], [0], "bo", label="burn 2")
    ax2.set_aspect("equal"); ax2.set_xlabel("x (km)"); ax2.set_ylabel("y (km)")
    ax2.set_title("LEO to GEO Hohmann transfer"); ax2.legend(loc="upper right", fontsize=8); ax2.grid(alpha=0.3)
    fig2.tight_layout(); fig2.savefig(OUT / "hohmann_integrated.png", dpi=120)

    # Figure 3: plane change cost against speed
    ang = np.linspace(0, 60, 100)
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    for name, v in (("LEO", vleo), ("GEO", vgeo), ("transfer apogee", vapo)):
        ax3.plot(ang, plane_change_dv(v, ang) / 1e3, label=f"{name}, v = {v / 1e3:.2f} km/s")
    ax3.set_xlabel("plane change (degrees)"); ax3.set_ylabel("delta-v (km/s)")
    ax3.set_title("Plane changes are cheap where you are slow"); ax3.legend(); ax3.grid(alpha=0.3)
    fig3.tight_layout(); fig3.savefig(OUT / "plane_change.png", dpi=120)

    # Mini-project
    m1, m2 = R_EARTH + 500e3, R_EARTH + 20200e3
    hm = hohmann(m1, m2)
    cm = check_hohmann(m1, m2)
    print(f"mini-project: r1 {m1 / 1e3:.0f}, r2 {m2 / 1e3:.0f}: dv1 {hm['dv1']:.1f}, dv2 {hm['dv2']:.1f}, total {hm['total']:.1f} m/s, time {hm['time'] / 3600:.3f} h; arrival r {cm['r_end'] / 1e3:.2f} km, angle {cm['angle']:.2f}, circle err {cm['circle_err']:.1f} m")
    print("ratio r2/r1", m2 / m1)

## Coding tasks
1. **Easy.** Use `hohmann` to print the burns for 500 km to 20200 km altitude, then check against the mini-project success criteria.
2. **Stretch.** Use `phasing_time(6778e3, 7178e3, 60)` and compare with `phasing_formula`. Predict first whether the time doubles when the lead angle doubles.
3. **Explain.** Explain why the Oberth effect means a burn is best done at the lowest point of the orbit, and why a plane change is best done at the highest.

## Assessment
- **A. Conceptual.** Why are both Hohmann burns along the velocity (tangential), and why is the flight time half an ellipse period?
- **B. Numerical.** Find the Hohmann burns and the transfer time between the circular orbits of radius 6778 km and 7178 km (a small step, for example before docking).
- **C. Graph and simulation interpretation.** In the bi-elliptic graph, for which radius ratios is Hohmann cheaper, and what does the 11.94 crossing mean for LEO to GEO?
- **D. Design and optimisation challenge.** Design: a satellite is launched into a low orbit inclined 28.5 degrees and must be put into the equatorial geostationary orbit. Compare three plans for the plane change: in the low orbit, at the geostationary orbit, and combined with the second burn at the transfer apogee. Which would you choose and what is the trade-off?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    r1, r2 = 6778e3, 42164e3
    h = L.hohmann(r1, r2)
    checks.append((f"worked example: LEO to GEO dv1 2.398, dv2 1.457, total 3.854 km/s (got {h['dv1'] / 1e3:.3f}, {h['dv2'] / 1e3:.3f}, {h['total'] / 1e3:.3f})",
                   abs(h["dv1"] / 1e3 - 2.398) < 5e-4 and abs(h["dv2"] / 1e3 - 1.457) < 5e-4 and abs(h["total"] / 1e3 - 3.854) < 5e-4))
    checks.append((f"worked example: transfer takes 5.291 h on an ellipse with a = 24471 km (got {h['time'] / 3600:.3f}, {h['a_t'] / 1e3:.0f})",
                   abs(h["time"] / 3600 - 5.291) < 5e-4 and abs(h["a_t"] / 1e3 - 24471) < 0.5))
    vp_t, va_t = L.vis_viva(r1, h["a_t"]), L.vis_viva(r2, h["a_t"])
    checks.append((f"worked example speeds: circular LEO 7.669, transfer perigee 10.066, transfer apogee 1.618, circular GEO 3.075 km/s (got {L.circular_speed(r1) / 1e3:.3f}, {vp_t / 1e3:.3f}, {va_t / 1e3:.3f}, {L.circular_speed(r2) / 1e3:.3f})",
                   abs(L.circular_speed(r1) / 1e3 - 7.669) < 5e-4 and abs(vp_t / 1e3 - 10.066) < 5e-4 and abs(va_t / 1e3 - 1.618) < 5e-4 and abs(L.circular_speed(r2) / 1e3 - 3.075) < 5e-4))
    sm = L.hohmann(6778e3, 7178e3)
    checks.append((f"small step 6778 km to 7178 km: 109.1 + 107.6 = 216.7 m/s in 2900.5 s (got {sm['dv1']:.1f}, {sm['dv2']:.1f}, {sm['total']:.1f}, {sm['time']:.1f})",
                   abs(sm["dv1"] - 109.1) < 0.05 and abs(sm["dv2"] - 107.6) < 0.05 and abs(sm["total"] - 216.7) < 0.05 and abs(sm["time"] - 2900.5) < 0.05))
    c = L.check_hohmann(r1, r2)
    checks.append((f"integration from burn 1 arrives at 42164 km (within 1 km), opposite side (180 degrees) (got {c['r_end'] / 1e3:.2f} km, {c['angle']:.2f})",
                   abs(c["r_end"] - r2) < 1e3 and abs(c["angle"] - 180) < 0.01))
    checks.append((f"integrated arrival speed equals the vis-viva apogee speed 1618 m/s (got {c['v_end']:.1f})",
                   abs(c["v_end"] - L.vis_viva(r2, h["a_t"])) < 0.5 and abs(c["v_end"] - 1618) < 1))
    checks.append((f"after burn 2 the orbit stays circular to within 1 m over a lap (got {c['circle_err']:.2f} m)", c["circle_err"] < 1.0))
    z = L.hohmann(7e6, 7e6)
    checks.append(("edge: transfer between equal radii costs nothing and takes one half orbit", z["total"] < 1e-9 and abs(z["time"] - 0.5 * L.period(7e6)) < 1e-6))
    d = L.hohmann(r2, r1)
    checks.append(("going down costs the same total, with both burns backwards", d["dv1"] < 0 and d["dv2"] < 0 and abs(d["total"] - h["total"]) < 1e-6))

    # bi-elliptic
    vals = {x: L.cost_ratio(x) for x in (5, 10, 15, 20, 50)}
    checks.append(("ratio 5: bi-elliptic costs 1.249 times Hohmann; ratio 10: 1.029; ratio 15: 0.972",
                   abs(vals[5] - 1.2488) < 5e-4 and abs(vals[10] - 1.0291) < 5e-4 and abs(vals[15] - 0.9719) < 5e-4))
    scan = np.arange(2.0, 30.0, 0.01)
    first = scan[next(i for i, x in enumerate(scan) if L.cost_ratio(x) < 1)]
    checks.append((f"scan: bi-elliptic first wins at a ratio of about 11.94 (scan {first:.2f}, bisection {L.crossover_ratio():.3f})",
                   abs(first - 11.94) < 0.011 and abs(L.crossover_ratio() - 11.94) < 0.005))
    b = L.bielliptic(r1, r2, r2)
    checks.append(("edge: a bi-elliptic transfer with rb = r2 is the Hohmann transfer", abs(b["total"] - h["total"]) < 1e-6))
    b = L.bielliptic(r1, r2, 20 * r2)
    checks.append((f"LEO to GEO is a ratio of 6.2, so bi-elliptic loses: 4.427 km/s and 32.9 days (got {b['total'] / 1e3:.3f}, {b['time'] / 86400:.1f})",
                   abs(b["total"] / 1e3 - 4.427) < 5e-4 and abs(b["time"] / 86400 - 32.9) < 0.05))

    # plane change
    vleo, vgeo = L.circular_speed(r1), L.circular_speed(r2)
    vapo = L.vis_viva(r2, h["a_t"])
    pc = [L.plane_change_dv(v, 28.5) / 1e3 for v in (vleo, vgeo, vapo)]
    checks.append((f"28.5 degrees costs 3.775 (LEO), 1.514 (GEO), 0.797 (transfer apogee) km/s (got {pc[0]:.3f}, {pc[1]:.3f}, {pc[2]:.3f})",
                   abs(pc[0] - 3.775) < 5e-4 and abs(pc[1] - 1.514) < 5e-4 and abs(pc[2] - 0.797) < 5e-4))
    checks.append(("hand calculation: a 60 degree turn costs exactly v, a 0 degree turn costs nothing",
                   abs(L.plane_change_dv(1000.0, 60) - 1000.0) < 1e-9 and L.plane_change_dv(1000.0, 0) == 0))
    comb = L.combined_burn_dv(vapo, vgeo, 28.5) / 1e3
    sep = (abs(vgeo - vapo) + L.plane_change_dv(vgeo, 28.5)) / 1e3
    checks.append((f"combined apogee burn 1.824 km/s beats circularising then turning, 2.970 km/s (got {comb:.3f}, {sep:.3f})",
                   abs(comb - 1.824) < 5e-4 and abs(sep - 2.970) < 5e-4 and comb < sep))
    opt_a = (L.plane_change_dv(vleo, 28.5) + h["total"]) / 1e3
    opt_b = (h["dv1"] + L.combined_burn_dv(vapo, vgeo, 28.5)) / 1e3
    opt_c = (h["total"] + L.plane_change_dv(vgeo, 28.5)) / 1e3
    checks.append((f"design: total delta-v to GEO with 28.5 degrees to remove: 7.629 (turn in LEO), 5.368 (turn at GEO), 4.222 (turn at apogee) km/s (got {opt_a:.3f}, {opt_c:.3f}, {opt_b:.3f})",
                   abs(opt_a - 7.629) < 5e-4 and abs(opt_c - 5.368) < 5e-4 and abs(opt_b - 4.222) < 5e-4 and opt_b < opt_c < opt_a))
    checks.append(("combined burn with no turn is just the speed difference", abs(L.combined_burn_dv(3000.0, 4000.0, 0) - 1000.0) < 1e-9))

    # Oberth
    low, high = L.energy_gain(vleo, 1000.0) / 1e6, L.energy_gain(1000.0, 1000.0) / 1e6
    checks.append((f"Oberth: 1000 m/s adds 8.169 MJ/kg at LEO speed and 1.500 MJ/kg at 1000 m/s (got {low:.3f}, {high:.3f})",
                   abs(low - 8.169) < 5e-4 and abs(high - 1.5) < 5e-4))
    checks.append(("energy gain equals v dv + dv^2/2", abs(L.energy_gain(5000.0, 300.0) - (5000.0 * 300.0 + 0.5 * 300.0**2)) < 1e-6))

    # phasing
    tn, tf = L.phasing_time(6778e3, 7178e3, 30), L.phasing_formula(6778e3, 7178e3, 30)
    checks.append((f"phasing: chaser at 6778 km catches a target 30 degrees ahead at 7178 km in 5615.5 s (numeric {tn:.1f}, formula {tf:.1f})",
                   abs(tn - tf) < 0.5 and abs(tn - 5615.5) < 0.5))

    # finite burn
    acc20 = L.finite_burn(r1, h["dv1"] / 20.0, 20.0)
    acc2 = L.finite_burn(r1, h["dv1"] / 2.0, 2.0)
    acc05 = L.finite_burn(r1, h["dv1"] / 0.5, 0.5)
    checks.append((f"finite burn apogees: 42111 km (20 m/s^2), 38026 km (2 m/s^2), 23023 km (0.5 m/s^2) (got {acc20 / 1e3:.0f}, {acc2 / 1e3:.0f}, {acc05 / 1e3:.0f})",
                   abs(acc20 / 1e3 - 42111) < 3 and abs(acc2 / 1e3 - 38026) < 3 and abs(acc05 / 1e3 - 23023) < 3))
    c2, c05 = L.finite_burn_cost(r1, r2, 2.0), L.finite_burn_cost(r1, r2, 0.5)
    e2, e05 = (c2 - h["dv1"]) / h["dv1"], (c05 - h["dv1"]) / h["dv1"]
    checks.append((f"gravity loss: a 2 m/s^2 burn needs 3.5% more delta-v, a 0.5 m/s^2 burn 35.9% more (got {e2:.1%}, {e05:.1%})",
                   abs(e2 - 0.035) < 0.001 and abs(e05 - 0.359) < 0.001))
    checks.append(("weaker engines lose more", acc05 < acc2 < acc20 < r2 and e05 > e2 > 0))

    # mini-project
    m1, m2 = L.R_EARTH + 500e3, L.R_EARTH + 20200e3
    hm = L.hohmann(m1, m2)
    cm = L.check_hohmann(m1, m2)
    checks.append((f"mini-project: 500 km to 20200 km altitude: dv1 1984.8, dv2 1390.3, total 3375.1 m/s, 2.989 h (got {hm['dv1']:.1f}, {hm['dv2']:.1f}, {hm['total']:.1f}, {hm['time'] / 3600:.3f})",
                   abs(hm["dv1"] - 1984.8) < 0.1 and abs(hm["dv2"] - 1390.3) < 0.1 and abs(hm["total"] - 3375.1) < 0.1 and abs(hm["time"] / 3600 - 2.989) < 5e-4))
    checks.append((f"mini-project: integrated arrival radius 26571 km within 1 km, 180 degrees (got {cm['r_end'] / 1e3:.2f}, {cm['angle']:.2f})",
                   abs(cm["r_end"] - m2) < 1e3 and abs(cm["angle"] - 180) < 0.01))
    checks.append(("mini-project: a bi-elliptic route with rb = 3 r2 costs more than Hohmann at this small ratio",
                   L.bielliptic(m1, m2, 3 * m2)["total"] > hm["total"]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
